In [ ]:
import json
import numpy as np
import pandas as pd

# Define the 9 spatial offsets for directions 0 to 8
# Format: (delta_x, delta_y)
# Adjust the 0.15 multiplier to make the spacing tighter or wider
spacing = 1
direction_offsets = {
    0: (0, 0),  # Center
    1: (spacing, 0),  # Right
    2: (0, spacing),  # Top
    4: (0, -spacing),  # Bottom
    3: (-spacing, 0),  # Left
    5: (spacing, spacing),  # Top-Right
    6: (-spacing, spacing),  # Top-Left
    7: (-spacing, -spacing),  # Bottom-Left
    8: (spacing, -spacing),  # Bottom-Right
}

# Load the JSON data
with open("milestone02.json", "r") as f:
    raw_data = json.load(f)

processed_timesteps = {}

for timestep, points in raw_data.items():
    df = pd.DataFrame(points, columns=["x", "y", "direction", "value"])

    # Map the direction integer to its respective (dx, dy) offset
    df["dx"] = df["direction"].map(lambda d: direction_offsets.get(d, (0, 0))[0])
    df["dy"] = df["direction"].map(lambda d: direction_offsets.get(d, (0, 0))[1])

    # Calculate the new split coordinate positions
    df["x_split"] = df["x"] + df["dx"]
    df["y_split"] = df["y"] + df["dy"]

    processed_timesteps[timestep] = df


print(f"Processed {len(processed_timesteps)} timesteps with split directional points.")
timesteps = sorted(list(processed_timesteps.keys()))

Processed 100 timesteps with split directional points.


In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from ipywidgets import interact, IntSlider

# --- USER CONFIGURATION ---
# Choose which timestep to show if you aren't using the interactive slider
default_timestep = timesteps[0] 

def plot_timestep(timestep_name):
    # Retrieve the DataFrame for the current timestep
    df = processed_timesteps[timestep_name]
    
    # 1. Calculate vector components (u, v) based on your offsets and values
    # We normalize dx and dy so the "value" acts as the true speed magnitude
    norm = np.hypot(df['dx'], df['dy'])
    
    # Avoid division by zero for direction 0 (Center)
    u_arr = np.where(norm > 0, df['value'] * (df['dx'] / norm), 0.0)
    v_arr = np.where(norm > 0, df['value'] * (df['dy'] / norm), 0.0)
    
    x_arr = df['x'].to_numpy()
    y_arr = df['y'].to_numpy()
    
    # 2. Create the uniform grid required by streamplot
    # 100x100 resolution is a good balance between speed and quality
    grid_x, grid_y = np.meshgrid(
        np.linspace(x_arr.min(), x_arr.max(), 100),
        np.linspace(y_arr.min(), y_arr.max(), 100)
    )
    
    # 3. Interpolate the scattered data onto the grid
    grid_u = griddata((x_arr, y_arr), u_arr, (grid_x, grid_y), method='linear')
    grid_v = griddata((x_arr, y_arr), v_arr, (grid_x, grid_y), method='linear')
    
    # Compute overall speed on the grid for line color-coding
    speed = np.sqrt(grid_u**2 + grid_v**2)
    
    # 4. Plotting
    fig, ax = plt.subplots(figsize=(9, 7), dpi=100)
    
    strm = ax.streamplot(
        grid_x, grid_y, grid_u, grid_v, 
        color=speed,              
        cmap='plasma',           
        linewidth=1.5,            
        density=1.3               
    )
    
    # Add a colorbar and styling
    fig.colorbar(strm.lines, ax=ax, label='Velocity Magnitude (value)')
    
    # Optional: Lightly dot the original coordinate positions
    ax.scatter(x_arr, y_arr, color='gray', alpha=0.3, s=10, label='Grid Points')
    
    ax.set_title(f"Streamplot - Timestep: {timestep_name}", fontsize=14, pad=15)
    ax.set_xlabel("X Coordinate")
    ax.set_ylabel("Y Coordinate")
    ax.grid(True, linestyle='--', alpha=0.3)
    
    plt.show()

# --- CHOOSE YOUR RUN MODE ---

# OPTION A: Interactive Slider (Recommended for Jupyter Notebooks)
# This lets you scrub through all your loaded timesteps sequentially.
def update_plot(index):
    plot_timestep(timesteps[index])

interact(
    update_plot, 
    index=IntSlider(min=0, max=len(timesteps)-1, step=1, value=0, description='Timestep Iter:')
);

# OPTION B: Static Plot
# If you just want a single quick plot without widgets, uncomment the line below:
# plot_timestep(default_timestep)